<a href="https://colab.research.google.com/github/MalakMostafa-16003651/nyc-housing-transportation-analysis/blob/feature%2Fdata-integration/data_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from getpass import getpass

token = getpass("Paste your NEW GitHub token here: ")

Paste your NEW GitHub token here: ··········


In [24]:
import base64
import subprocess

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

result = subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader=AUTHORIZATION: basic {auth}",
        "clone",
        "https://github.com/MalakMostafa-16003651/nyc-housing-transportation-analysis.git"
    ],
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)


Cloning into 'nyc-housing-transportation-analysis'...



In [25]:
%cd nyc-housing-transportation-analysis

/content/nyc-housing-transportation-analysis/nyc-housing-transportation-analysis


In [26]:
!git branch -a

* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/feature/data-integration
  remotes/origin/main


In [27]:
!git checkout feature/data-integration

branch 'feature/data-integration' set up to track 'origin/feature/data-integration'.
Switched to a new branch 'feature/data-integration'


In [28]:
!git branch

* feature/data-integration
  main


In [29]:
!find . -maxdepth 3 -type f

./Housing_Data_Cleaning.ipynb
./.git/logs/HEAD
./.git/info/exclude
./.git/hooks/post-update.sample
./.git/hooks/commit-msg.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-receive.sample
./.git/hooks/pre-push.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/applypatch-msg.sample
./.git/hooks/update.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/sendemail-validate.sample
./.git/config
./.git/description
./.git/HEAD
./.git/index
./.git/packed-refs
./df_housing_clean.csv
./README.md


In [30]:
import pandas as pd

df_housing_clean = pd.read_csv("df_housing_clean.csv")
df_housing_clean.head()

,project_id,project_name,project_start_date,project_completion_date,building_id,house_number,street_name,borough,zip_code,bbl,...,_3_br_units,_4_br_units,_5_br_units,_6_br_units,unknown_br_units,counted_rental_units,counted_homeownership_units,all_counted_units,total_units,year
0,68872,177-06 WEXFORD TERRACE,2026-06-30,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43,2026
1,72436,ST. NICK'S. 819 GRAND STREET,2026-06-30,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187,2026
2,72449,BREMOND KING DAVIS.YR15.FY26,2026-06-30,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25,2026
3,72449,BREMOND KING DAVIS.YR15.FY26,2026-06-30,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16,2026
4,72449,BREMOND KING DAVIS.YR15.FY26,2026-06-30,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15,2026


In [31]:
df_housing_clean[
    ["project_id", "building_id", "borough", "zip_code", "all_counted_units", "total_units", "year"]
].dtypes

,0
project_id,int64
building_id,float64
borough,object
zip_code,float64
all_counted_units,float64
total_units,int64
year,int64


In [32]:
df_housing_clean["zip_code"] = (
    df_housing_clean["zip_code"]
    .astype("Int64")
    .astype("string")
)

df_housing_clean[["zip_code", "year"]].head()

,zip_code,year
0,11432,2026
1,11211,2026
2,10030,2026
3,10030,2026
4,10030,2026


In [33]:
df_housing_clean[["zip_code", "borough", "year"]].isna().sum()

,0
zip_code,1958
borough,0
year,0


In [34]:
total_rows = len(df_housing_clean)
missing_zip = df_housing_clean["zip_code"].isna().sum()

print("Total rows:", total_rows)
print("Missing ZIP rows:", missing_zip)
print("Missing ZIP %:", round((missing_zip / total_rows) * 100, 2))

Total rows: 9551
Missing ZIP rows: 1958
Missing ZIP %: 20.5


In [35]:
df_housing_clean.groupby("year")["zip_code"].apply(
    lambda x: x.isna().sum()
)

,zip_code
year,
2014,302
2015,122
2016,145
2017,147
2018,114
2019,108
2020,51
2021,114
2022,201


In [36]:
zip_missing_by_year = df_housing_clean.groupby("year").agg(
    total_rows=("zip_code", "size"),
    missing_zip=("zip_code", lambda x: x.isna().sum())
)

zip_missing_by_year["missing_percent"] = (
    zip_missing_by_year["missing_zip"] /
    zip_missing_by_year["total_rows"] * 100
).round(2)

zip_missing_by_year

,total_rows,missing_zip,missing_percent
year,,,
2014,734,302,41.14
2015,694,122,17.58
2016,722,145,20.08
2017,703,147,20.91
2018,691,114,16.50
2019,816,108,13.24
2020,608,51,8.39
2021,780,114,14.62
2022,670,201,30.00


In [37]:
total_units = df_housing_clean["all_counted_units"].sum()
missing_zip_units = df_housing_clean.loc[
    df_housing_clean["zip_code"].isna(),
    "all_counted_units"
].sum()

print("Total affordable units:", total_units)
print("Affordable units in rows with missing ZIP:", missing_zip_units)
print(
    "Percentage of affordable units lost:",
    round((missing_zip_units / total_units) * 100, 2)
)

Total affordable units: 320946.0
Affordable units in rows with missing ZIP: 3351.0
Percentage of affordable units lost: 1.04


In [38]:
df_housing_integration = df_housing_clean.dropna(
    subset=["zip_code"]
).copy()

print("Rows before:", len(df_housing_clean))
print("Rows after:", len(df_housing_integration))
print("Rows removed:", len(df_housing_clean) - len(df_housing_integration))

Rows before: 9551
Rows after: 7593
Rows removed: 1958


In [39]:
df_housing_agg = (
    df_housing_integration
    .groupby(["zip_code", "year", "borough"], as_index=False)
    .agg(
        housing_building_count=("building_id", "nunique"),
        housing_project_count=("project_id", "nunique"),
        affordable_units=("all_counted_units", "sum"),
        total_housing_units=("total_units", "sum")
    )
)

df_housing_agg.head()

,zip_code,year,borough,housing_building_count,housing_project_count,affordable_units,total_housing_units
0,10001,2014,Manhattan,2,2,194.0,939
1,10001,2015,Manhattan,3,3,188.0,366
2,10001,2016,Manhattan,1,1,174.0,174
3,10001,2017,Manhattan,7,1,1460.0,1878
4,10001,2018,Manhattan,2,2,52.0,167


In [40]:
duplicates = df_housing_agg.duplicated(
    subset=["zip_code", "year"],
    keep=False
).sum()

print("Duplicate ZIP-year rows:", duplicates)

Duplicate ZIP-year rows: 6


In [41]:
df_housing_agg[
    df_housing_agg.duplicated(
        subset=["zip_code", "year"],
        keep=False
    )
].sort_values(["zip_code", "year"])

,zip_code,year,borough,housing_building_count,housing_project_count,affordable_units,total_housing_units
464,10463,2014,Bronx,2,1,18.0,18
465,10463,2014,Manhattan,1,1,317.0,317
473,10463,2024,Bronx,9,9,610.0,690
474,10463,2024,Manhattan,2,1,6.0,16
475,10463,2025,Bronx,3,3,104.0,142
476,10463,2025,Manhattan,2,2,46.0,151


In [42]:
df_housing_agg = (
    df_housing_integration
    .groupby(["zip_code", "year"], as_index=False)
    .agg(
        borough=("borough", lambda x: " / ".join(sorted(x.dropna().unique()))),
        housing_building_count=("building_id", "nunique"),
        housing_project_count=("project_id", "nunique"),
        affordable_units=("all_counted_units", "sum"),
        total_housing_units=("total_units", "sum")
    )
)

print(
    "Duplicate ZIP-year rows:",
    df_housing_agg.duplicated(["zip_code", "year"]).sum()
)

df_housing_agg[df_housing_agg["zip_code"] == "10463"].head()

Duplicate ZIP-year rows: 0


,zip_code,year,borough,housing_building_count,housing_project_count,affordable_units,total_housing_units
464,10463,2014,Bronx / Manhattan,3,2,335.0,335
465,10463,2017,Bronx,2,2,275.0,275
466,10463,2018,Bronx,2,2,19.0,85
467,10463,2019,Bronx,3,3,199.0,318
468,10463,2020,Bronx,1,1,14.0,46


In [43]:
print("Before aggregation:")
print(df_housing_integration["all_counted_units"].sum())
print(df_housing_integration["total_units"].sum())

print("\nAfter aggregation:")
print(df_housing_agg["affordable_units"].sum())
print(df_housing_agg["total_housing_units"].sum())

Before aggregation:
317595.0
441985

After aggregation:
317595.0
441985


In [44]:
df_housing_agg.to_csv("df_housing_agg.csv", index=False)

In [45]:
!git status

On branch feature/data-integration
Your branch is up to date with 'origin/feature/data-integration'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	df_housing_agg.csv

nothing added to commit but untracked files present (use "git add" to track)


In [46]:
!git add df_housing_agg.csv

In [48]:
!git config user.name "MalakMostafa-16003651"

In [49]:
!git config user.email "malak.afifi@student.giu-uni.de"

In [50]:
!git commit -m "add aggregated housing data for integration"

[feature/data-integration 530f3c4] add aggregated housing data for integration
 1 file changed, 1156 insertions(+)
 create mode 100644 df_housing_agg.csv


In [51]:
!git push origin feature/data-integration

fatal: could not read Username for 'https://github.com': No such device or address


In [52]:
import base64
import subprocess

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

result = subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader=AUTHORIZATION: basic {auth}",
        "push",
        "origin",
        "feature/data-integration"
    ],
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)


To https://github.com/MalakMostafa-16003651/nyc-housing-transportation-analysis.git
   ee9bf1d..530f3c4  feature/data-integration -> feature/data-integration



In [53]:
!git status

On branch feature/data-integration
Your branch is up to date with 'origin/feature/data-integration'.

nothing to commit, working tree clean
